# Jazz Visualizations

End-to-end analysis of the jazz Wikipedia extract.

| Section | What you get |
|---|---|
| 1. Genre breakdown | Sunburst hierarchy · genre×decade heatmap · co-occurrence matrix |
| 2. Artist popularity | Albums-per-artist · birth decade bars · instruments · labels |
| 3. Collaboration graph | Album personnel network → Plotly force layout · Louvain communities |
| 4. TF-IDF clustering | Block-parallel dump fetch → UMAP scatter · cluster treemap · similar artists |

**Kernel**: `Python (venv-gpu)` — cuML on GPU if available, sklearn + umap-learn on CPU otherwise.

In [1]:
# ── GPU / CPU library selection ────────────────────────────────────────────
# Switch the kernel to "Python 3 (ipykernel)" if you see import errors below.
# Launch from the repo: uv run jupyter lab
try:
    import cuml  # type: ignore[import-untyped]
    from cuml.cluster import KMeans
    from cuml.manifold import UMAP
    GPU = True
    print(f"✓ cuML {cuml.__version__} — UMAP + KMeans on GPU")
except ImportError:
    try:
        from sklearn.cluster import KMeans
        from umap import UMAP
        GPU = False
        print("⚠ cuML not found — CPU fallback (sklearn + umap-learn)")
    except ModuleNotFoundError as _e:
        raise SystemExit(
            f"\n❌ Missing: {_e}\n\n"
            "You're on a kernel that lacks sklearn / umap-learn.\n"
            "Fix: switch to the 'Python 3 (ipykernel)' kernel,\n"
            "     or launch Jupyter from the repo venv:\n\n"
            "         uv run jupyter lab\n"
        ) from _e

✓ cuML 26.04.000 — UMAP + KMeans on GPU


In [2]:
from __future__ import annotations

import bz2
import os
import re
import sys
import warnings
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

# Suppress tqdm's false-positive IProgress warning (fires during kernel
# startup before the widget comm channel is open; does not affect functionality)
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", category=FutureWarning)

import networkx as nx
import mwparserfromhell
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
from sqlalchemy import create_engine

# ── repo paths ─────────────────────────────────────────────────────────────
REPO = Path("../").resolve()
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")

from wiki_dumps.dumps.preprocess import load_raw_index, raw_paths
from wiki_dumps.settings import get_settings
from wiki_dumps.text_utils import strip_wiki_article_text

settings = get_settings()

# ── config ─────────────────────────────────────────────────────────────────
DB_PATH     = REPO / "data/databases/jazz.db"
DUMP_DIR    = REPO / settings.dump_dir
N_JOBS      = settings.n_jobs
N_WORKERS   = os.cpu_count() or 8 if N_JOBS < 0 else N_JOBS
N_CLUSTERS  = 10   # KMeans k for TF-IDF clustering
MAX_ARTISTS = None  # None = all; set an int to prototype on a subset
GRAPH_TOP_N = 300   # max nodes in collaboration graph
MIN_COLLAB  = 1     # min shared albums to draw an edge (personnel field is sparse)

print(f"DB:       {DB_PATH}")
print(f"Dump dir: {DUMP_DIR}")
print(f"Workers:  {N_WORKERS}")

DB:       /home/benwright/Documents/GitHub/wiki/data/databases/jazz.db
Dump dir: /home/benwright/Documents/GitHub/wiki/data/dumps
Workers:  32


In [3]:
# ── load tables ────────────────────────────────────────────────────────────
engine  = create_engine(f"sqlite:///{DB_PATH}")
artists = pd.read_sql_table("jazz_artists", engine)
albums  = pd.read_sql_table("jazz_albums",  engine)
genres  = pd.read_sql_table("jazz_genres",  engine)
engine.dispose()

if MAX_ARTISTS:
    artists = artists.head(MAX_ARTISTS)

print(f"artists={len(artists):,}  albums={len(albums):,}  genres={len(genres):,}")
artists.head(3)

artists=10,881  albums=1,386  genres=62


,id,page_id,title,full_name,birth_year,death_year,nationality,instruments,genres,active_years_start,active_years_end,labels
0,1,7668,Charles Mingus,Charles Mingus,1922.0,1979.0,"Los Angeles, California, U.S.",Double bass|piano|vocals,Jazz|hard bop|bebop|avant-garde jazz|post-bop|...,1943.0,None,Atlantic|Candid|Columbia|Debut|Impulse!|Mercur...
1,2,4010,Bing Crosby,Bing Crosby,1903.0,1977.0,None,None,Traditional pop|jazz|easy listening|big band,1923.0,None,Decca|Columbia|RCA Victor|Brunswick|Reprise|Ca...
2,3,8176,Dave Brubeck,None,1920.0,2012.0,None,Piano,Jazz\n Cool jazz\n West Coast jazz\n Third stream,2012.0,None,Fantasy\n Columbia\n Legacy\n Sony\n Decca\n A...


In [4]:
# ── build dump page-ID → block index (used by §4 TF-IDF) ──────────────────
DUMP_PATH  = sorted(DUMP_DIR.glob("*-pages-articles-multistream.xml.bz2"))[-1]
INDEX_PATH = Path(str(DUMP_PATH).replace(
    "-pages-articles-multistream.xml.bz2",
    "-pages-articles-multistream-index.txt.bz2",
))
RAW_PATH, IDX_PATH = raw_paths(DUMP_PATH)
assert RAW_PATH.exists(), (
    f"Preprocessed raw file not found: {RAW_PATH}\n"
    "Run: uv run wiki dump preprocess"
)

raw_index = load_raw_index(IDX_PATH)

# Cache keyed by dump mtime so it auto-invalidates on re-download
_mtime = int(INDEX_PATH.stat().st_mtime)
_cache = INDEX_PATH.with_suffix(f".{_mtime}.pid_blk.npz")

if _cache.exists():
    _data   = np.load(_cache)
    pid_arr = _data["pid_arr"]
    blk_arr = _data["blk_arr"]
    print(f"Loaded pid/blk index from cache  ({_cache.name})")
else:
    for _old in INDEX_PATH.parent.glob(INDEX_PATH.stem + ".*.pid_blk.npz"):
        _old.unlink()
    _pids, _blks, block_idx, prev_offset = [], [], -1, -1
    with bz2.open(INDEX_PATH, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            fc = line.index(":")
            sc = line.index(":", fc + 1)
            offset  = int(line[:fc])
            page_id = int(line[fc + 1 : sc])
            if offset != prev_offset:
                block_idx  += 1
                prev_offset = offset
            _pids.append(page_id)
            _blks.append(block_idx)
    pid_arr = np.array(_pids, dtype=np.int64)
    blk_arr = np.array(_blks, dtype=np.int32)
    order   = np.argsort(pid_arr, kind="stable")
    pid_arr = pid_arr[order]
    blk_arr = blk_arr[order]
    del _pids, _blks, order
    np.savez_compressed(_cache, pid_arr=pid_arr, blk_arr=blk_arr)
    print(f"Built and cached pid/blk index  ({_cache.name})")

print(f"Index: {len(pid_arr):,} pages in {blk_arr.max()+1:,} blocks")

Loaded pid/blk index from cache  (enwiki-20260301-pages-articles-multistream-index.txt.1777727529.pid_blk.npz)
Index: 25,432,678 pages in 254,363 blocks


---
## 1  Genre Breakdown

Genre data comes from three places:
- `jazz_artists.genres` — pipe-separated genres per artist (infobox)
- `jazz_albums.genres`  — pipe-separated genres per album (infobox)
- `jazz_genres`         — dedicated genre table with parent links

In [5]:
_LIST_TMPLS = frozenset({"hlist", "flatlist", "plainlist", "unbulleted list", "ubl"})

# Top-level umbrella genres — not useful as subgenre labels, excluded everywhere
_GENRE_EXCLUDE = frozenset({"jazz", "jazz music", "popular music", "music"})


def parse_wiki_list(raw: str) -> list[str]:
    """Parse a raw wikitext list value → plain-text items."""
    wc = mwparserfromhell.parse(raw)
    items: list[str] = []
    for tmpl in wc.filter_templates():
        if str(tmpl.name).strip().lower() in _LIST_TMPLS:
            for param in tmpl.params:
                if not param.showkey:
                    item = mwparserfromhell.parse(str(param.value)).strip_code().strip()
                    if item:
                        items.append(item)
    if not items:
        plain = wc.strip_code().strip()
        items = [i.strip() for i in re.split(r"[,|\n*]+", plain) if i.strip()]
    return items


def _drop_umbrella(s: pd.Series) -> pd.Series:
    """Filter a string Series, dropping entries in _GENRE_EXCLUDE (case-insensitive)."""
    return s[~s.str.lower().isin(_GENRE_EXCLUDE)]

In [6]:
# ── top subgenres by artist count ─────────────────────────────────────────
artist_genre_counts = (
    artists["genres"]
    .dropna()
    .str.split("|")
    .explode()
    .str.strip()
    .str.title()
    .pipe(_drop_umbrella)
    .value_counts()
    .head(25)
    .reset_index()
    .rename(columns={"genres": "genre", "count": "n_artists"})
)

fig = px.bar(
    artist_genre_counts,
    x="n_artists", y="genre",
    orientation="h",
    title="Top subgenres by artist count",
    labels={"n_artists": "# artists", "genre": ""},
    color="n_artists",
    color_continuous_scale="Blues",
    width=850, height=620,
)
fig.update_layout(coloraxis_showscale=False, yaxis_autorange="reversed")
fig.show()

In [7]:
# ── top subgenres by album count ──────────────────────────────────────────
album_genre_counts = (
    albums["genres"]
    .dropna()
    .str.split("|")
    .explode()
    .str.strip()
    .str.title()
    .pipe(_drop_umbrella)
    .value_counts()
    .head(25)
    .reset_index()
    .rename(columns={"genres": "genre", "count": "n_albums"})
)

fig = px.bar(
    album_genre_counts,
    x="n_albums", y="genre",
    orientation="h",
    title="Top subgenres by album count",
    labels={"n_albums": "# albums", "genre": ""},
    color="n_albums",
    color_continuous_scale="Teal",
    width=850, height=620,
)
fig.update_layout(coloraxis_showscale=False, yaxis_autorange="reversed")
fig.show()

In [8]:
# ── subgenre × birth-decade heatmap ───────────────────────────────────────
_gd = (
    artists[["birth_year", "genres"]]
    .dropna()
    .assign(
        decade=lambda df: (df["birth_year"].astype(int) // 10 * 10).astype(str) + "s",
        genre=lambda df: df["genres"].str.split("|"),
    )
    .explode("genre")
    .assign(genre=lambda df: df["genre"].str.strip().str.title())
    .dropna(subset=["genre"])
    .loc[lambda df: ~df["genre"].str.lower().isin(_GENRE_EXCLUDE)]
)

_top_genres_hm = _gd["genre"].value_counts().head(18).index.tolist()
_valid_decades = sorted(_gd["decade"].unique())

pivot = (
    _gd[_gd["genre"].isin(_top_genres_hm)]
    .groupby(["genre", "decade"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=_valid_decades, fill_value=0)
)

fig = px.imshow(
    pivot,
    title="Subgenre presence across artist birth decades",
    color_continuous_scale="Blues",
    aspect="auto",
    labels={"x": "Birth decade", "y": "Genre", "color": "# artists"},
    width=1050, height=560,
)
fig.update_xaxes(tickangle=0)
fig.show()

In [9]:
# ── subgenre co-occurrence (Jaccard similarity) ────────────────────────────
_genre_lists = (
    artists["genres"]
    .dropna()
    .str.split("|")
    .apply(lambda xs: [
        x.strip().title() for x in xs
        if x.strip() and x.strip().lower() not in _GENRE_EXCLUDE
    ])
    .tolist()
)

mlb = MultiLabelBinarizer()
G_mat = mlb.fit_transform(_genre_lists).astype(float)
genres_ordered = mlb.classes_

_top_n_cooc = 20
_top_idx = G_mat.sum(axis=0).argsort()[::-1][:_top_n_cooc]
G_sub = G_mat[:, _top_idx]
genres_sub = genres_ordered[_top_idx]

cooc = G_sub.T @ G_sub
diag = cooc.diagonal()
union = diag[:, None] + diag[None, :] - cooc
cooc_norm = np.divide(cooc, union, where=union > 0, out=np.zeros_like(cooc))
np.fill_diagonal(cooc_norm, 1.0)

fig = px.imshow(
    cooc_norm,
    x=genres_sub, y=genres_sub,
    title="Subgenre co-occurrence — Jaccard similarity (artist infoboxes)",
    color_continuous_scale="RdBu",
    zmin=0, zmax=1,
    width=850, height=820,
)
fig.update_xaxes(tickangle=40)
fig.show()

In [10]:
# ── genre hierarchy sunburst ───────────────────────────────────────────────
# "Jazz" is the invisible root; subgenres radiate from it.
known_titles = set(genres["title"].str.strip())

sunburst_rows: list[dict] = [{"genre": "Jazz", "parent": ""}]

for _, row in genres.iterrows():
    child = str(row["title"]).strip()
    if child.lower() in _GENRE_EXCLUDE:
        continue
    if pd.isna(row["parent_genre"]) or not row["parent_genre"]:
        parent = "Jazz"
    else:
        candidates = parse_wiki_list(str(row["parent_genre"]))
        parent = next((p for p in candidates if p in known_titles and p != child), "Jazz")
    sunburst_rows.append({"genre": child, "parent": parent})

sb_df = pd.DataFrame(sunburst_rows)

# Artist count per subgenre — normalise both sides to lowercase so casing
# differences between the genres table and artist infoboxes don't cause misses.
_genre_artist_count_lc = (
    artists["genres"].dropna().str.split("|").explode()
    .str.strip().str.lower()
    .loc[lambda s: ~s.isin(_GENRE_EXCLUDE)]
    .value_counts().to_dict()
)
sb_df["n_artists"] = (
    sb_df["genre"].str.lower()
    .map(_genre_artist_count_lc)
    .fillna(3).astype(int)
)
sb_df.loc[sb_df["genre"] == "Jazz", "n_artists"] = 1  # root is a container

fig = px.sunburst(
    sb_df,
    names="genre",
    parents="parent",
    values="n_artists",
    title="Jazz subgenre hierarchy  (sized by artist count)",
    width=850, height=850,
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_traces(textinfo="label+percent entry", insidetextorientation="auto")
fig.show()

---
## 2  Artist Popularity

Popularity proxy: **album count** (# dedicated Wikipedia album pages per artist).  
Also covers birth/death decade distributions, instruments, and top labels.

In [11]:
# ── most-albumed artists ───────────────────────────────────────────────────
album_counts = (
    albums["artist"]
    .dropna()
    .str.strip()
    .value_counts()
    .reset_index()
    .rename(columns={"artist": "artist", "count": "n_albums"})
    .head(40)
)

fig = px.bar(
    album_counts,
    x="n_albums", y="artist",
    orientation="h",
    title="Most-covered jazz artists (Wikipedia album count)",
    labels={"n_albums": "# albums in DB", "artist": ""},
    color="n_albums",
    color_continuous_scale="Viridis",
    width=900, height=950,
)
fig.update_layout(coloraxis_showscale=False, yaxis_autorange="reversed")
fig.show()

In [12]:
# ── birth decade — living vs deceased ─────────────────────────────────────
decade_df = artists[["birth_year", "death_year"]].dropna(subset=["birth_year"]).copy()
decade_df["decade"] = (decade_df["birth_year"].astype(int) // 10 * 10).astype(int)
decade_df["status"] = decade_df["death_year"].isna().map({True: "Living", False: "Deceased"})

decade_counts = decade_df.groupby(["decade", "status"]).size().reset_index(name="count")

fig = px.bar(
    decade_counts,
    x="decade", y="count", color="status",
    barmode="stack",
    title="Jazz artists by birth decade",
    labels={"decade": "Birth decade", "count": "# artists"},
    color_discrete_map={"Living": "#4C78A8", "Deceased": "#F58518"},
    width=1000, height=460,
)
fig.update_xaxes(tickmode="array", tickvals=sorted(decade_df["decade"].unique()))
fig.show()

In [13]:
# ── instrument popularity ──────────────────────────────────────────────────
instrument_counts = (
    artists["instruments"]
    .dropna()
    .str.split("|")
    .explode()
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .value_counts()
    .head(25)
    .reset_index()
    .rename(columns={"instruments": "instrument", "count": "n_artists"})
)

fig = px.bar(
    instrument_counts,
    x="n_artists", y="instrument",
    orientation="h",
    title="Top instruments among jazz artists",
    color="n_artists",
    color_continuous_scale="Teal",
    labels={"n_artists": "# artists", "instrument": ""},
    width=800, height=650,
)
fig.update_layout(coloraxis_showscale=False, yaxis_autorange="reversed")
fig.show()

In [14]:
# ── top record labels ──────────────────────────────────────────────────────
label_counts = (
    artists["labels"]
    .dropna()
    .str.split("|")
    .explode()
    .str.strip()
    .value_counts()
    .head(25)
    .reset_index()
    .rename(columns={"labels": "label", "count": "n_artists"})
)

fig = px.bar(
    label_counts,
    x="n_artists", y="label",
    orientation="h",
    title="Top record labels (by artist count)",
    color="n_artists",
    color_continuous_scale="Purples",
    labels={"n_artists": "# artists", "label": ""},
    width=800, height=650,
)
fig.update_layout(coloraxis_showscale=False, yaxis_autorange="reversed")
fig.show()

In [15]:
# ── label × genre heatmap ─────────────────────────────────────────────────
# Which labels sign which genres?  Top 15 labels × top 12 genres.
_top_labels = label_counts.head(15)["label"].tolist()
_top_genres_lg = artist_genre_counts.head(12)["genre"].tolist()

_lg = (
    artists[["labels", "genres"]]
    .dropna()
    .assign(
        label=lambda df: df["labels"].str.split("|"),
        genre=lambda df: df["genres"].str.split("|"),
    )
    .explode("label")
    .explode("genre")
    .assign(
        label=lambda df: df["label"].str.strip(),
        genre=lambda df: df["genre"].str.strip().str.title(),
    )
    .query("label in @_top_labels and genre in @_top_genres_lg")
)

_lg_pivot = (
    _lg.groupby(["label", "genre"]).size().unstack(fill_value=0)
    .reindex(index=_top_labels, columns=_top_genres_lg, fill_value=0)
)

fig = px.imshow(
    _lg_pivot,
    title="Label × genre heatmap — how many artists per cell",
    color_continuous_scale="Blues",
    aspect="auto",
    labels={"x": "Genre", "y": "Label", "color": "# artists"},
    width=950, height=620,
)
fig.update_xaxes(tickangle=35)
fig.show()

---
## 3  Artist Collaboration Graph

Edges = **album personnel co-appearances** — two artists share an edge for  
every album whose `==Personnel==` section lists them both.  
Weight = number of shared albums.

Node colour = Louvain community.  Node size = weighted degree.

> **Note**: requires re-running the jazz extraction after the `jazz.py` fix  
> that now captures the `==Personnel==` article section.

In [16]:
def _extract_wikilinks(raw: str | None) -> list[str]:
    """Return all wikilink titles from raw wikitext (strips display text)."""
    if not raw:
        return []
    wc = mwparserfromhell.parse(raw)
    return [str(link.title).strip().split("|")[0].strip() for link in wc.filter_wikilinks()]


# Known artist page titles for filtering co-appearance links
artist_titles = set(artists["title"].str.strip())

# Count how many albums each pair of artists appear on together
edge_weights: dict[tuple[str, str], int] = defaultdict(int)
artist_credit_count: dict[str, int] = defaultdict(int)

for _, row in albums.iterrows():
    if pd.isna(row["personnel"]) or not row["personnel"]:
        continue
    links = _extract_wikilinks(str(row["personnel"]))
    crew  = list(dict.fromkeys(l for l in links if l in artist_titles))
    for i, a in enumerate(crew):
        artist_credit_count[a] += 1
        for b in crew[i + 1 :]:
            key = (min(a, b), max(a, b))
            edge_weights[key] += 1

n_with_personnel = albums["personnel"].notna().sum()
print(f"Albums with personnel:          {n_with_personnel:,} / {len(albums):,}")
print(f"Co-appearance edges (raw):      {len(edge_weights):,}")
print(f"Artists with personnel credits: {len(artist_credit_count):,}")

if n_with_personnel == 0:
    print("\n⚠ personnel is all NULL — re-run the jazz extraction:")
    print("   uv run wiki extract run jazz")

Albums with personnel:          1,118 / 1,386
Co-appearance edges (raw):      10,512
Artists with personnel credits: 1,535


In [17]:
# ── build NetworkX graph ───────────────────────────────────────────────────
G = nx.Graph()
for (a, b), w in edge_weights.items():
    G.add_edge(a, b, weight=w)

if G.number_of_edges() == 0:
    raise RuntimeError(
        "No co-appearance edges — personnel field is empty.\n"
        "Re-run the jazz extraction: uv run wiki extract run jazz"
    )

# Keep largest connected component, then trim to GRAPH_TOP_N by weighted degree
lcc = max(nx.connected_components(G), key=len)
G   = G.subgraph(lcc).copy()

if G.number_of_nodes() > GRAPH_TOP_N:
    _wdeg = {n: sum(d for _, _, d in G.edges(n, data="weight", default=1)) for n in G}
    _keep = {n for n, _ in sorted(_wdeg.items(), key=lambda x: x[1], reverse=True)[:GRAPH_TOP_N]}
    G = G.subgraph(_keep).copy()

print(f"Graph: {G.number_of_nodes():,} nodes · {G.number_of_edges():,} edges")
degrees = [d for _, d in G.degree()]
print(f"Degree: mean={np.mean(degrees):.1f}  max={max(degrees)}")

Graph: 300 nodes · 4,779 edges
Degree: mean=31.9  max=110


In [18]:
# ── Louvain communities + spring layout ───────────────────────────────────
communities = list(nx.algorithms.community.greedy_modularity_communities(G, weight="weight"))
community_map = {node: cid for cid, comm in enumerate(communities) for node in comm}
print(f"Louvain communities: {len(communities)}")
for cid, comm in enumerate(communities):
    print(f"  Community {cid:2d}: {len(comm):3d} artists")

print("\nComputing spring layout…")
pos = nx.spring_layout(G, weight="weight", seed=42, k=1.2 / np.sqrt(G.number_of_nodes()))
print("Done.")

# Pull primary genre + birth year for hover
title_to_genre = dict(zip(artists["title"], artists["genres"].fillna("")))
title_to_birth = dict(zip(artists["title"], artists["birth_year"].fillna("")))

Louvain communities: 6
  Community  0: 103 artists
  Community  1:  98 artists
  Community  2:  53 artists
  Community  3:  30 artists
  Community  4:  11 artists
  Community  5:   5 artists

Computing spring layout…
Done.


In [39]:
# ── Plotly collaboration graph ─────────────────────────────────────────────
_palette = px.colors.qualitative.Bold + px.colors.qualitative.Plotly

# Edge trace (all edges as one trace for speed)
ex, ey, ew = [], [], []
for u, v, data in G.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    ex += [x0, x1, None]
    ey += [y0, y1, None]
    ew.append(data.get("weight", 1))

edge_trace = go.Scatter(
    x=ex, y=ey,
    mode="lines",
    line=dict(width=0.6, color="rgba(160,160,160,0.35)"),
    hoverinfo="none",
    showlegend=False,
)

# Node traces — one per community (keeps legend clean)
node_traces = []
for cid, comm in enumerate(communities):
    nodes_in = [n for n in comm if n in pos]
    if not nodes_in:
        continue
    xs    = [pos[n][0] for n in nodes_in]
    ys    = [pos[n][1] for n in nodes_in]
    sizes = [10 + G.degree(n) * 0.15 for n in nodes_in]
    texts = [
        (
            f"<b>{n}</b><br>"
            f"degree: {G.degree(n)}<br>"
            f"genre: {title_to_genre.get(n,'').split('|')[0] or '—'}<br>"
            f"born: {int(title_to_birth.get(n,0)) if title_to_birth.get(n) else '—'}"
        )
        for n in nodes_in
    ]
    node_traces.append(go.Scatter(
        x=xs, y=ys,
        mode="markers",
        marker=dict(
            size=sizes,
            color=_palette[cid % len(_palette)],
            line=dict(width=0.7, color="white"),
        ),
        text=texts,
        hovertemplate="%{text}<extra></extra>",
        name=f"Community {cid}  ({len(nodes_in)})",
    ))

fig = go.Figure(data=[edge_trace] + node_traces)
fig.update_layout(
    title=(
        f"Jazz artist collaboration network — {G.number_of_nodes()} artists · "
        f"{G.number_of_edges()} edges · {len(communities)} communities"
    ),
    hovermode="closest",
    showlegend=True,
    legend=dict(title="Community", font_size=11, itemsizing="constant"),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor="rgba(15,15,20,1)",
    paper_bgcolor="rgba(15,15,20,1)",
    font_color="#ddd",
    width=1200, height=850,
)
fig.show()

In [20]:
# ── top bridging artists (betweenness centrality) ─────────────────────────
print("Computing betweenness centrality…")
bc = nx.betweenness_centrality(G, weight="weight", normalized=True)
top_bc = sorted(bc.items(), key=lambda x: x[1], reverse=True)[:20]

bc_df = pd.DataFrame(top_bc, columns=["artist", "betweenness"])
bc_df["community"] = bc_df["artist"].map(community_map)
bc_df["degree"]    = bc_df["artist"].map(dict(G.degree()))
bc_df["genre"]     = bc_df["artist"].map(lambda n: title_to_genre.get(n, "").split("|")[0])

fig = px.bar(
    bc_df,
    x="betweenness", y="artist",
    orientation="h",
    title="Top bridging artists by betweenness centrality",
    color="community", color_discrete_sequence=_palette,
    hover_data=["degree", "genre"],
    labels={"betweenness": "Betweenness", "artist": ""},
    width=900, height=600,
)
fig.update_layout(yaxis_autorange="reversed", showlegend=False)
fig.show()

Computing betweenness centrality…


---
## 4  TF-IDF Clustering

Fetches the full Wikipedia article text for every artist, strips markup with  
`wiki_dumps.text_utils.strip_wiki_article_text`, then runs:

```
TF-IDF (1-2 gram, 3 000 features)
    → Truncated SVD (128 components)
    → UMAP (2-D)
    → KMeans (N_CLUSTERS)
```

Use the **GPU kernel** for the UMAP + KMeans steps on large artist counts.

In [21]:
# ── group page IDs by raw-dump block ──────────────────────────────────────
page_ids = artists["page_id"].tolist()

_block_to_pids: dict[int, list[int]] = defaultdict(list)
for pid in page_ids:
    pos_idx = int(np.searchsorted(pid_arr, pid))
    if pos_idx < len(pid_arr) and pid_arr[pos_idx] == pid:
        _block_to_pids[int(blk_arr[pos_idx])].append(pid)

print(f"{len(page_ids):,} pages span {len(_block_to_pids):,} unique blocks "
      f"(avg {len(page_ids)/max(len(_block_to_pids),1):.1f} pages/block)")


def _fetch_block_strip(args: tuple[int, int, list[int], str]) -> dict[int, str]:
    """Read one raw block and return stripped prose for each requested page ID.

    Fully self-contained — safe for spawn-based multiprocessing.
    """
    from wiki_dumps.parse.stream import iter_pages_from_xml_bytes
    from wiki_dumps.text_utils import strip_wiki_article_text

    offset, length, pids, raw_path = args
    want = set(pids)
    results: dict[int, str] = {}
    with open(raw_path, "rb") as fh:
        fh.seek(offset)
        xml_bytes = fh.read(length)
    for page in iter_pages_from_xml_bytes(xml_bytes):
        if page.page_id in want:
            try:
                results[page.page_id] = strip_wiki_article_text(page.wikitext)
            except Exception:  # noqa: BLE001
                results[page.page_id] = page.wikitext
            if len(results) == len(want):
                break
    return results


_raw_path_str = str(RAW_PATH)
_block_args = [
    (raw_index[blk][0], raw_index[blk][1], pids, _raw_path_str)
    for blk, pids in _block_to_pids.items()
]

print(f"Fetching {len(_block_args):,} blocks across {N_WORKERS} workers…")
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    _results = list(ex.map(_fetch_block_strip, _block_args, chunksize=20))

pid_to_text: dict[int, str] = {}
for r in _results:
    pid_to_text.update(r)

corpus = [pid_to_text.get(int(pid), "") for pid in page_ids]
n_empty = sum(1 for t in corpus if not t)
avg_len = float(np.mean([len(t) for t in corpus if t]))
print(f"Done — {len(corpus):,} docs | {n_empty:,} empty | avg {avg_len:,.0f} chars/doc")

10,881 pages span 9,687 unique blocks (avg 1.1 pages/block)
Fetching 9,687 blocks across 32 workers…
Done — 10,881 docs | 0 empty | avg 5,185 chars/doc


In [22]:
# ── TF-IDF vectorization with jazz-specific stopwords ─────────────────────
_JAZZ_STOPWORDS: set[str] = {
    # Wikipedia boilerplate
    "redirect", "external", "links", "contents", "references", "isbn",
    "retrieved", "archived", "thumb", "right", "left", "edit", "category",
    "section", "article", "page",
    # Jazz article near-universals (non-discriminative)
    "jazz", "music", "musical", "musician", "musicians",
    "band", "ensemble", "orchestra", "quintet", "quartet", "trio",
    "recorded", "recording", "recordings", "record", "records",
    "released", "release", "releases",
    "album", "albums", "song", "songs", "track", "tracks",
    "single", "singles",
    "played", "playing", "play", "plays",
    "performance", "performed", "perform", "performances",
    "known", "became", "career", "began", "early", "life",
    "born", "died", "death",
    "american", "british", "new", "york", "city",
    "united", "states",
    "also", "often", "one", "two", "three",
    "first", "second", "third", "last",
    "well", "noted", "include", "including", "included",
    "style", "styles", "influenced", "influence", "influences",
    "work", "works", "working",
}

all_stop = list(ENGLISH_STOP_WORDS | _JAZZ_STOPWORDS)

vectorizer = TfidfVectorizer(
    max_features=3000,
    min_df=0.01,
    max_df=0.80,
    sublinear_tf=True,
    strip_accents="unicode",
    stop_words=all_stop,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]{2,}\b",
)
X = vectorizer.fit_transform(corpus)
print(f"TF-IDF matrix: {X.shape[0]:,} docs × {X.shape[1]:,} terms  ({X.nnz:,} non-zeros)")

TF-IDF matrix: 10,881 docs × 3,000 terms  (1,745,983 non-zeros)


In [23]:
# ── Truncated SVD: sparse → dense ──────────────────────────────────────────
SVD_COMPONENTS = 128
svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=42)
X_svd = svd.fit_transform(X)
print(f"SVD variance explained: {svd.explained_variance_ratio_.sum():.1%}")

SVD variance explained: 28.7%


In [24]:
# ── UMAP → 2-D ────────────────────────────────────────────────────────────
umap_kwargs: dict = dict(n_components=2, n_neighbors=15, min_dist=0.05, random_state=42, verbose=True)
if not GPU:
    umap_kwargs["n_jobs"] = N_JOBS

reducer = UMAP(**umap_kwargs)
embedding = reducer.fit_transform(X_svd)
print(f"Embedding shape: {embedding.shape}")

[2026-05-31 19:22:43.403] [CUML] [debug] Computing KNN Graph
[2026-05-31 19:22:43.423] [CUML] [debug] Computing fuzzy simplicial set
Embedding shape: (10881, 2)


In [25]:
# ── KMeans clustering ──────────────────────────────────────────────────────
km_kwargs: dict = dict(n_clusters=N_CLUSTERS, random_state=42)
if not GPU:
    km_kwargs["n_init"] = "auto"

km = KMeans(**km_kwargs)
labels = km.fit_predict(X_svd)  # cluster on SVD, not 2-D
if hasattr(labels, "to_numpy"):  # cuML returns cuDF series
    labels = labels.to_numpy()

artists = artists.copy()
artists["cluster"] = labels
print("Cluster sizes:")
print(pd.Series(labels).value_counts().sort_index().to_string())

Cluster sizes:
0    1512
1     392
2    1154
3     471
4    2149
5    1339
6     232
7    1644
8    1089
9     899


In [26]:
# ── interactive UMAP scatter ───────────────────────────────────────────────
emb   = np.array(embedding)
terms = np.array(vectorizer.get_feature_names_out())

cluster_top_terms: dict[int, str] = {}
for cid in range(N_CLUSTERS):
    mask = labels == cid
    if not mask.any():
        continue
    centroid = np.asarray(X[mask].mean(axis=0)).ravel()
    top3 = terms[centroid.argsort()[::-1][:3]]
    cluster_top_terms[cid] = " · ".join(top3)

plot_df = artists[["title", "cluster", "birth_year", "genres", "instruments"]].copy()
plot_df["x"]             = emb[:, 0]
plot_df["y"]             = emb[:, 1]
plot_df["cluster_str"]   = plot_df["cluster"].astype(str)
plot_df["top_terms"]     = plot_df["cluster"].map(cluster_top_terms)
plot_df["primary_genre"] = plot_df["genres"].fillna("").str.split("|").str[0].str.strip()

fig = px.scatter(
    plot_df, x="x", y="y",
    color="cluster_str",
    hover_data=["title", "birth_year", "primary_genre", "instruments", "top_terms"],
    title=f"Jazz artists — TF-IDF + UMAP  ({N_CLUSTERS} clusters)",
    width=1200, height=800,
    opacity=0.55,
    color_discrete_sequence=px.colors.qualitative.Bold,
    labels={"color": "cluster"},
)
fig.update_traces(marker_size=4)
fig.update_layout(
    legend_title_text="cluster",
    plot_bgcolor="rgba(15,15,20,1)",
    paper_bgcolor="rgba(15,15,20,1)",
    font_color="#ddd",
)
fig.show()

In [27]:
# ── cluster treemap with top terms ─────────────────────────────────────────
treemap_rows = []
for cid in range(N_CLUSTERS):
    mask = labels == cid
    if not mask.any():
        continue
    centroid = np.asarray(X[mask].mean(axis=0)).ravel()
    top_terms_str = " · ".join(terms[centroid.argsort()[::-1][:8]])
    treemap_rows.append({
        "cluster":   f"Cluster {cid}",
        "size":      int(mask.sum()),
        "top_terms": top_terms_str,
    })

tm_df = pd.DataFrame(treemap_rows)
fig2 = px.treemap(
    tm_df,
    path=["cluster"],
    values="size",
    custom_data=["top_terms", "size"],
    title="Cluster sizes — jazz artists (TF-IDF)",
    color="size",
    color_continuous_scale="Blues",
    width=1200, height=540,
)
fig2.update_traces(
    texttemplate="<b>%{label}</b><br>%{customdata[1]} artists<br><br><i>%{customdata[0]}</i>",
    textfont_size=12,
    hovertemplate="<b>%{label}</b><br>%{customdata[1]} artists<br>%{customdata[0]}<extra></extra>",
)
fig2.update_layout(coloraxis_showscale=False)
fig2.show()

In [28]:
# ── top TF-IDF terms per cluster ───────────────────────────────────────────
TOP_N = 20
rows = []
for cid in range(N_CLUSTERS):
    mask = labels == cid
    if not mask.any():
        continue
    centroid = np.asarray(X[mask].mean(axis=0)).ravel()
    top_idx  = centroid.argsort()[::-1][:TOP_N]
    rows.append({
        "cluster":    cid,
        "size":       int(mask.sum()),
        "top_terms":  ", ".join(terms[top_idx]),
    })

cluster_summary = (
    pd.DataFrame(rows)
    .sort_values("size", ascending=False)
    .set_index("cluster")
)
with pd.option_context("display.max_colwidth", None):
    display(cluster_summary)

,size,top_terms
cluster,,
4,2149,"pianists, composers, century, people, male, composer, living people, living, university, piano, pianist, classical, award, school, year, college, festival, awards, alumni, studied"
7,1644,"guitarists, bass, guitar, drums, john, guitarist, live, saxophone, tenor, male, piano, people, rock, listing, drummer, discography, living, century, review, living people"
0,1512,"male, century, deaths, trumpeters, orleans, saxophonists, births deaths, century male, worked, male male, grove, chicago, led, bands, louis, dictionary, grove dictionary, drummers, kernfeld, trombonists"
5,1339,"film, time, year, years, later, singer, best, television, radio, love, singers, series, award, world, hit, chart, featured, school, artists, appeared"
2,1154,"singers, women, women singers, singer, singers century, century, singing, people, living, living people, year, pop, century women, singers singers, songwriters, singer songwriters, vocal, sang, vocalist, best"
8,1089,"note, blue note, blue, live, artists, leader, male, soul, sideman, john, century, drummers, time, muse, columbia, joe, saxophonists, discography leader, david, school"
9,899,"verve, rca, capitol, basie, male, rca victor, columbia, prestige, big, victor, atlantic, count, blues, jones, joe, jimmy, count basie, charlie, benny, blue"
3,471,"norwegian, century norwegian, norwegian male, oslo, norway, norwegian composers, trondheim, composers, century, johansen, municipality, curling, curling legs, spellemannprisen, male, legs, bergen, terje, births living, ole"
1,392,"double bassists, bassists, double, bassists male, bassists century, male double, century double, bass, male, bassist, century, double bassist, double bass, century male, worked, male male, live, john, deaths, blue"


In [29]:
# ── find most similar artists by cosine similarity in SVD space ─────────────
QUERY = "Miles Davis"   # ← change to any artist title in the table
TOP_K = 15

idx = artists.index[artists["title"] == QUERY].tolist()
if not idx:
    print(f"{QUERY!r} not found — try one of: {artists['title'].sample(5).tolist()}")
else:
    query_vec = X_svd[idx[0] : idx[0] + 1]
    sims = cosine_similarity(query_vec, X_svd).ravel()
    top_idx = sims.argsort()[::-1][1 : TOP_K + 1]  # skip self

    result = artists.iloc[top_idx][["title", "cluster", "birth_year", "genres", "instruments"]].copy()
    result["similarity"] = sims[top_idx].round(4)
    print(f"Most similar to: {QUERY!r}")
    display(result)

Most similar to: 'Miles Davis'


,title,cluster,birth_year,genres,instruments,similarity
175,Bill Evans,5,1929.0,Jazz|modal jazz|third stream|cool jazz|smooth ...,Piano,0.9075
56,Charlie Parker,5,1920.0,Jazz\n bebop,Alto and tenor saxophone,0.8738
76,Thelonious Monk,5,1917.0,Jazz|bebop,Piano,0.8683
23,Herbie Hancock,5,1940.0,Jazz|post-bop|modal jazz|jazz fusion|jazz-funk...,Keyboards|keytar|vocoder|synthesizer,0.8648
289,Sonny Rollins,5,1930.0,Jazz|hard bop,Tenor saxophone|soprano saxophone,0.8555
8,Frank Zappa,5,1940.0,Rock|progressive rock|blues|experimental|jazz|...,None,0.8477
172,Bud Powell,5,1924.0,Jazz|bebop,Piano,0.8475
0,Charles Mingus,5,1922.0,Jazz|hard bop|bebop|avant-garde jazz|post-bop|...,Double bass|piano|vocals,0.8442
132,Art Tatum,5,1909.0,Jazz|stride,Piano,0.8383
624,Ahmad Jamal,5,1930.0,Jazz|hard bop|modal jazz|cool jazz|post-bop,Piano,0.8317
